# EDA и эксперименты: классификация обращений в поддержку

Цель ноутбука — показать разведочный анализ данных, обучить несколько моделей и обосновать выбор финальной модели для FastAPI-сервиса.

Проект решает задачу многоклассовой классификации пользовательского обращения по тексту. Модель предсказывает категорию тикета, например проблему с оплатой, доставкой, возвратом, доступом к аккаунту, техническую ошибку или общий вопрос.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline

# Ноутбук рассчитан на запуск из project/notebooks/main.ipynb.
# Также он будет работать, если открыть его из корня проекта.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import clean_text

DATA_PATH = PROJECT_ROOT / "data" / "tickets.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.25

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_PATH: {DATA_PATH}")


## 1. Загрузка данных

Проверяем, что датасет существует, содержит нужные колонки и подходит для обучения модели классификации.


In [ ]:
df = pd.read_csv(DATA_PATH)

print("Размер датасета:", df.shape)
print("Колонки:", list(df.columns))

df.head(10)


In [ ]:
required_columns = {"text", "category"}
missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(f"В датасете отсутствуют обязательные колонки: {missing_columns}")

print("Пропуски по колонкам:")
print(df[["text", "category"]].isna().sum())

print("
Количество дублей:", int(df.duplicated().sum()))


## 2. Разведочный анализ данных

Смотрим баланс классов, длину текстов и примеры обращений. Это помогает понять, насколько задача подходит для простой NLP-модели и нет ли явных проблем в данных.


In [ ]:
class_counts = df["category"].value_counts().sort_values(ascending=False)
class_share = (class_counts / len(df)).round(3)

eda_summary = pd.DataFrame({
    "rows": class_counts,
    "share": class_share,
})

eda_summary


In [ ]:
plt.figure(figsize=(10, 5))
class_counts.plot(kind="bar")
plt.title("Распределение обращений по категориям")
plt.xlabel("Категория")
plt.ylabel("Количество примеров")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
df["text_len_chars"] = df["text"].astype(str).str.len()
df["text_len_words"] = df["text"].astype(str).str.split().str.len()

df[["text_len_chars", "text_len_words"]].describe().round(2)


In [ ]:
plt.figure(figsize=(8, 5))
df["text_len_words"].plot(kind="hist", bins=20)
plt.title("Распределение длины обращений в словах")
plt.xlabel("Количество слов")
plt.ylabel("Количество обращений")
plt.tight_layout()
plt.show()


In [ ]:
examples = (
    df.groupby("category", group_keys=False)
    .apply(lambda part: part.sample(2, random_state=RANDOM_STATE))
    [["category", "text"]]
    .reset_index(drop=True)
)

examples


### Выводы по EDA

- Датасет небольшой, поэтому для первой версии разумно использовать простые интерпретируемые модели.
- Классы распределены достаточно равномерно, поэтому accuracy можно смотреть, но основная метрика — `f1_macro`, так как она учитывает качество по всем классам одинаково.
- Тексты короткие, поэтому полезны не только word n-grams, но и character n-grams: они помогают при коротких фразах и опечатках.


## 3. Подготовка train/test split

Используем стратифицированное разбиение, чтобы в train и test сохранились пропорции классов.


In [ ]:
X = df["text"].astype(str)
y = df["category"].astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Classes:", sorted(y.unique()))


## 4. Эксперименты с моделями

Сравним несколько вариантов:

1. `DummyClassifier` — baseline без анализа текста.
2. `CountVectorizer + LogisticRegression` — простой текстовый baseline.
3. `TF-IDF word n-grams + LogisticRegression` — улучшенная модель по словам.
4. `TF-IDF word + char n-grams + LogisticRegression` — финальная модель проекта.


In [ ]:
models = {
    "dummy_most_frequent": Pipeline([
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]),
    "count_word_logreg": Pipeline([
        ("vectorizer", CountVectorizer(preprocessor=clean_text, ngram_range=(1, 2), min_df=1)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "tfidf_word_logreg": Pipeline([
        ("vectorizer", TfidfVectorizer(preprocessor=clean_text, analyzer="word", ngram_range=(1, 2), min_df=1)),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "tfidf_word_char_logreg": Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                preprocessor=clean_text,
                analyzer="word",
                ngram_range=(1, 2),
                min_df=1,
            )),
            ("char_tfidf", TfidfVectorizer(
                preprocessor=clean_text,
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=1,
            )),
        ])),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

results = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    results.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, predictions),
        "f1_macro": f1_score(y_test, predictions, average="macro"),
        "f1_weighted": f1_score(y_test, predictions, average="weighted"),
    })
    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
results_df


In [ ]:
results_path = ARTIFACTS_DIR / "experiment_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Сохранили таблицу экспериментов: {results_path}")


In [ ]:
plt.figure(figsize=(10, 5))
plot_data = results_df.set_index("model")[["accuracy", "f1_macro", "f1_weighted"]]
plot_data.plot(kind="bar")
plt.title("Сравнение моделей по метрикам")
plt.xlabel("Модель")
plt.ylabel("Значение метрики")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 5. Детальная оценка финальной модели

Финальной моделью считаем `tfidf_word_char_logreg`, потому что она совпадает с моделью в `src/train.py` и использует одновременно word-level и char-level признаки.


In [ ]:
final_model_name = "tfidf_word_char_logreg"
final_model = trained_models[final_model_name]
final_predictions = final_model.predict(X_test)

print(classification_report(y_test, final_predictions, zero_division=0))


In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, final_predictions, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df


In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion matrix финальной модели")
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
plt.yticks(range(len(labels)), labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


In [ ]:
errors = pd.DataFrame({
    "text": X_test.values,
    "true_category": y_test.values,
    "predicted_category": final_predictions,
})

errors = errors[errors["true_category"] != errors["predicted_category"]]
print("Количество ошибок:", len(errors))
errors.head(20)


## 6. Проверка нескольких примеров

Проверим, как финальная модель работает на коротких пользовательских фразах, похожих на реальные обращения.


In [ ]:
sample_requests = [
    "не могу войти в аккаунт, пароль не подходит",
    "с карты списали деньги два раза",
    "курьер не привез заказ",
    "хочу вернуть деньги за заказ",
    "приложение выдает ошибку при оплате",
    "подскажите условия доставки",
]

sample_predictions = final_model.predict(sample_requests)

sample_result = pd.DataFrame({
    "text": sample_requests,
    "predicted_category": sample_predictions,
})

if hasattr(final_model, "predict_proba"):
    probabilities = final_model.predict_proba(sample_requests).max(axis=1)
    sample_result["confidence"] = probabilities.round(3)

sample_result


## 7. Итоговый вывод

По результатам экспериментов финальной моделью выбрана `TF-IDF word + char n-grams + LogisticRegression`.

Причины выбора:

- модель заметно лучше простого `DummyClassifier`, значит она действительно извлекает сигнал из текста;
- TF-IDF-признаки хорошо подходят для коротких текстовых обращений;
- word n-grams учитывают смысловые словосочетания;
- char n-grams помогают при коротких фразах и простых опечатках;
- Logistic Regression быстро обучается, интерпретируема и подходит для небольшого учебного сервиса.

Ограничения:

- датасет синтетический и небольшой;
- качество на реальных обращениях может быть ниже;
- для production-версии нужно собрать реальные данные, добавить мониторинг качества и регулярно переобучать модель.
